Day 2, Topic 4: np.take() and np.put()

## 📘 Day 2, Topic 4: `np.take()` and `np.put()`

### Why These Functions Exist
Fancy indexing with `arr[[i, j, k]]` always works on the **flattened** array unless you're explicit about axes. `np.take()` and `np.put()` are the **axis-aware, explicit** alternatives — cleaner and less error-prone for multi-dimensional arrays.

---

## `np.take()` — Axis-Aware Fancy Indexing

### Syntax
```python
np.take(array, indices, axis=None, mode='raise')
```

### axis=None (default) — Flattened Indexing
Treats the entire array as if it were 1D:
```python
arr3d = np.arange(24).reshape(2, 3, 4)
np.take(arr3d, [15, 12, 17], axis=None)   # flat positions 15, 12, 17
```

### axis=0 — Select Along First Dimension (rows/blocks)
```python
arr2d = np.arange(12).reshape(3, 4)
np.take(arr2d, [2, 2, 0], axis=0)   # rows 2, 2, 0 (can repeat!)
# → shape (3, 4) — three rows selected
```

### axis=1 — Select Columns
```python
np.take(arr2d, [1, 3, 0], axis=1)   # columns 1, 3, 0 (reordered)
```

### Out-of-Bounds Handling — The `mode` Parameter

| mode | Behavior | Example: index 5 on array of size 3 |
|------|----------|--------------------------------------|
| `'raise'` (default) | Raises `IndexError` | Error! |
| `'wrap'` | Wraps around cyclically | `5 % 3 = 2` → index 2 |
| `'clip'` | Clips to valid range | → index 2 (last valid) |

```python
arr = np.array([10, 20, 30])    # size 3, valid indices: 0, 1, 2
np.take(arr, [1, 3, 5, 10], mode='wrap')   # [20, 10, 30, 20]
#  index 3 → 3%3=0 → 10
#  index 5 → 5%3=2 → 30
#  index 10 → 10%3=1 → 20

np.take(arr, [1, 3, 5, 10], mode='clip')   # [20, 30, 30, 30]
#  all out-of-bounds → clipped to index 2 → 30
```

---

## `np.put()` — In-Place Modification via Flat Indices

### Syntax
```python
np.put(array, flat_indices, values)
```
`np.put()` always uses **flat (raveled) indices**, regardless of the array's shape. It modifies the array **in-place**.

```python
arr = np.array([10, 20, 30, 40, 50])
np.put(arr, [2, 0, 4], [99, 45, 55])
# arr → [45, 20, 99, 40, 55]
```

### Using `np.put()` on Multi-Dimensional Arrays
The flat index formula for shape `(d0, d1, d2)` is:
```
flat_index = i*(d1*d2) + j*d2 + k
```
```python
arr3d = np.arange(24).reshape(2, 3, 4)
# Target: block=1, row=2, col=3
flat = 1*(3*4) + 2*4 + 3   # = 23
np.put(arr3d, 23, 999)
```

### `np.ravel_multi_index()` — Convert (i,j,k) Coordinates to Flat Index
Instead of computing flat indices manually, use this helper:
```python
multi_indices = ([1, 0], [2, 1], [3, 0])   # (block, row, col) pairs
flat = np.ravel_multi_index(multi_indices, arr3d.shape)
# → [23, 4]    flat positions

np.put(arr3d, flat, [888, 777])
```

### Value Cycling in `np.put()`
If you provide fewer values than indices, NumPy **cycles** through them:
```python
arr = np.zeros(5)
np.put(arr, [0, 1, 2, 3, 4], [1, 2])
# → [1., 2., 1., 2., 1.]    values cycle: 1, 2, 1, 2, 1
```

---

### `np.take()` vs Fancy Indexing vs `np.put()` Summary

| | `arr[[i,j]]` | `np.take()` | `np.put()` |
|--|--|--|--|
| Axis support | Manual | Built-in `axis=` | No (always flat) |
| Out-of-bounds | Raises | `mode=` parameter | Raises |
| Modifies original | No (copy) | No (copy) | **Yes (in-place)** |
| Use case | Quick selection | Axis-aware selection | In-place update |

In [2]:
import numpy as np

In [2]:
arr = np.array([10, 20, 30, 40, 50])
indices = np.array([2, 3, 1, 0])

In [3]:
result = np.take(arr, indices)

In [4]:
result

array([30, 40, 20, 10])

In [7]:
arr2d = np.arange(12).reshape(3, 4)
print(arr2d)

[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]


In [8]:
row_index = np.array([2, 2, 0])

In [9]:
result = np.take(arr2d, row_index, axis=0)

In [10]:
result

array([[ 8,  9, 10, 11],
       [ 8,  9, 10, 11],
       [ 0,  1,  2,  3]])

In [12]:
col_index = np.array([1, 3, 0])
result = np.take(arr2d, col_index, axis=1)

In [13]:
result

array([[ 1,  3,  0],
       [ 5,  7,  4],
       [ 9, 11,  8]])

In [14]:
arr3d = np.arange(24).reshape(2, 3, 4)

In [15]:
arr3d

array([[[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]],

       [[12, 13, 14, 15],
        [16, 17, 18, 19],
        [20, 21, 22, 23]]])

In [16]:
block_index = np.array([1, 0, 0])
result = np.take(arr3d, block_index, axis=0)

In [18]:
result.shape

(3, 3, 4)

In [19]:
row_index = np.array([1, 2, 0])
result = np.take(arr3d, row_index, axis=1)

In [21]:
result.shape

(2, 3, 4)

In [22]:
col_index = np.array([1, 2, 0, 3])
result = np.take(arr3d, col_index, axis=2)

In [25]:
result.shape

(2, 3, 4)

In [26]:
flat_index = np.array([15, 12, 17])

In [27]:
result = np.take(arr3d, flat_index, axis=None)

In [28]:
result

array([15, 12, 17])

In [29]:
array = np.array([10, 20, 30])
indices = np.array([1, 3, 5, 10])

In [30]:
result = np.take(array, indices, mode='wrap')

In [31]:
result #[20, 10, 30, 20]

array([20, 10, 30, 20])

In [ ]:
array = np.array([10, 20, 30])
indices = np.array([1, 3, 5, 10])

In [32]:
result = np.take(array, indices, mode='clip')

In [33]:
result

array([20, 30, 30, 30])

In [34]:
array = np.array([10, 20, 30, 40, 50])
np.put(array, [2, 0, 4], [99, 45, 55])

In [35]:
array

array([45, 20, 99, 40, 55])

In [36]:
arr2d = np.arange(12).reshape(3, 4)
print(arr2d)

[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]


In [37]:
np.put(arr2d, [10, 11, 8], [456, 999, 212])

In [38]:
arr2d

array([[  0,   1,   2,   3],
       [  4,   5,   6,   7],
       [212,   9, 456, 999]])

In [39]:
arr = np.zeros(5)

In [40]:
np.put(arr, [0, 1, 2, 3, 4], [1, 2])

In [41]:
arr

array([1., 2., 1., 2., 1.])

In [ ]:
#flat_index = i * (dim1 * dim2) + j * dim2 + k

In [3]:
arr3d = np.arange(24).reshape(2, 3, 4)

In [4]:
arr3d

array([[[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]],

       [[12, 13, 14, 15],
        [16, 17, 18, 19],
        [20, 21, 22, 23]]])

In [7]:
#We want block 1, row 2, colum 3
flat_index = 1 * (3 * 4) + 2 * 4 + 3
np.put(arr3d, flat_index, 999) 

In [8]:
arr3d

array([[[  0,   1,   2,   3],
        [  4,   5,   6,   7],
        [  8,   9,  10,  11]],

       [[ 12,  13,  14,  15],
        [ 16,  17,  18,  19],
        [ 20,  21,  22, 999]]])

In [11]:
multi_indices = ([1, 0], [2, 1], [3, 0])   # (block, row, col) pairs
flat_indices = np.ravel_multi_index(multi_indices, arr3d.shape)
print(flat_indices)   # [23  4]

[23  4]


In [13]:
np.put(arr3d, flat_indices, [888, 777])
print(arr3d[1, 2, 3])   # 888
print(arr3d[0, 1, 0])   # 777

888
777


Task: You have a 3D array representing 3 batches × 4 rows × 5 columns.

In [36]:
def np_take_put(data):
    #Use np.take() to extract columns 0, 2, and 4 (in that order) from every batch and row. What is the shape of the result?
    col_index = np.array([0, 2, 4])
    result = np.take(data, col_index, axis=2)
    print(f"The shape of data after extracting column 0, 2 and 4: {result.shape}")
    #Use np.take() with axis=1 to extract rows 3, 0, and 1 from every batch. What is the shape?
    row_index = np.array([3, 0, 1])
    result = np.take(data, row_index, axis=1)
    print(f"The shape of data after extracting row 3, 0 and 1: {result.shape}")
    #Use np.put() to set the following positions to -1:
    #Batch 0, Row 2, Col 1
    #Batch 2, Row 3, Col 4
    #Batch 1, Row 0, Col 0
    multi_indices = ([0, 2, 1], [2, 3, 0], [1, 4, 0])
    flat_indices = np.ravel_multi_index(multi_indices, data.shape)
    np.put(data, flat_indices, -1)
    print(f"Updated data:\n{data}")
    #Use mode='wrap' with np.take() 
    #on a 1D array [10, 20, 30, 40] with indices [2, 5, 8, 11]. Predict the output before running.
    arr_1d = np.array([10, 20, 30, 40])
    result = np.take(arr_1d, [2, 5, 8, 1], mode='wrap')
    print("After applying wrap method on 1D aray: ",result) # 30, 20, 10, 20

In [37]:
data = np.arange(60).reshape(3, 4, 5)
np_take_put(data)

The shape of data after extracting column 0, 2 and 4: (3, 4, 3)
The shape of data after extracting row 3, 0 and 1: (3, 3, 5)
Updated data:
[[[ 0  1  2  3  4]
  [ 5  6  7  8  9]
  [10 -1 12 13 14]
  [15 16 17 18 19]]

 [[-1 21 22 23 24]
  [25 26 27 28 29]
  [30 31 32 33 34]
  [35 36 37 38 39]]

 [[40 41 42 43 44]
  [45 46 47 48 49]
  [50 51 52 53 54]
  [55 56 57 58 -1]]]
After applying wrap method on 1D aray:  [30 20 10 20]


In [38]:
import numpy as np

In [40]:
arr_3d = np.arange(24).reshape(2, 3, 4)
print(arr_3d)

[[[ 0  1  2  3]
  [ 4  5  6  7]
  [ 8  9 10 11]]

 [[12 13 14 15]
  [16 17 18 19]
  [20 21 22 23]]]


In [41]:
arr_3d[0, ...]

array([[ 0,  1,  2,  3],
       [ 4,  5,  6,  7],
       [ 8,  9, 10, 11]])

In [42]:
arr_3d[..., 2]

array([[ 2,  6, 10],
       [14, 18, 22]])

In [43]:
arr_3d[0, ..., 1:3]

array([[ 1,  2],
       [ 5,  6],
       [ 9, 10]])